# 03 — The Laws of Relative Entropy

`02/02` built the relative entropy $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ as the extra bits per symbol of coding with a model $q$ when the truth is $p$, proved that it is never negative (Gibbs' inequality), and showed the three ways it fails to be a distance. What it left open is why the course builds on it anyway. Today's answer is three laws that make it the currency of information. It adds up over independent symbols, which no bounded distance can do. It is jointly convex, and strictly so in its first argument, which total variation is not. And it never grows under processing, a law it shares with total variation (you proved the trace-distance version in `01/20`, exercise 4). One inequality proves the last two.

**Prerequisites:** `02/02_kl_divergence` (the definition, Gibbs' inequality, the infinite value off the support, the pair $p = (0.7, 0.2, 0.1)$, $q = (0.4, 0.4, 0.2)$ and its $0.2651$ bits), `02/01_shannon_entropy` (Jensen's inequality in §2, the chain rule of entropy in §3), `01/17_tensor_and_partial_trace` (product tables, joint tables and their marginals), `01/20_quantum_channels` (a stochastic matrix as a noisy channel on probability vectors), `01/15_fidelity_and_trace_distance` (total variation).

**What you'll be able to do**
- Prove additivity and the chain rule by hand, compute both on small tables, and explain why no bounded distance can be additive.
- Prove the log-sum inequality from Jensen's inequality, with its equality case, and derive joint convexity and the strict convexity that makes the KL projections of `03/11` and `04/08` unique.
- Prove the data-processing inequality $D(Tp\|Tq) \le D(p\|q)$ for every stochastic map $T$, and find by hand which merges lose nothing.
- Name the three relative entropies ahead and the law each one inherits from today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, mutual_information, shannon_entropy
from qot_course.infotheory.quantum import quantum_relative_entropy
from qot_course.quantum.density import density_matrix, maximally_mixed
from qot_course.quantum.states import KET_PLUS

rng = np.random.default_rng(0)   # random tables, mixtures and stochastic maps for the checks of sections 1-3
viz.use_course_style()

p = np.array([0.7, 0.2, 0.1])    # the truth of 02/02
q = np.array([0.4, 0.4, 0.2])    # the model of 02/02

def tv(p, q):
    # total variation of 01/15, written inline: half the l1 distance between two probability vectors
    return 0.5 * float(np.abs(np.asarray(p, dtype=float) - np.asarray(q, dtype=float)).sum())

## 1. Additive over independent symbols, and the chain rule

A code that writes two independent symbols one after the other spends the length of the first codeword plus the length of the second. Extra bits should therefore add up, and they do. Take two independent pairs, $p = p_A \otimes p_B$ and $q = q_A \otimes q_B$, the product tables of `01/17`, with entries $p_A(a)\,p_B(b)$. The logarithm of a product is a sum:
$$
D(p_A\!\otimes\! p_B\,\|\,q_A\!\otimes\! q_B) = \sum_{a,b} p_A(a)p_B(b)\Big[\log_2\tfrac{p_A(a)}{q_A(a)} + \log_2\tfrac{p_B(b)}{q_B(b)}\Big] = D(p_A\|q_A) + D(p_B\|q_B),
$$
because in each of the two terms the sum over the other variable is a sum of probabilities, equal to one. By induction, $n$ independent copies cost $n$ times as much: $D(p^{\otimes n}\|q^{\otimes n}) = n\,D(p\|q)$.

*By hand.* The dyadic source of `02/01` coded with the $2$-bit code costs $D(p\|u) = \tfrac14$ bit per symbol (`02/02` §1). Code pairs of symbols with the $4$-bit code for pairs: the cost per pair is $4$ bits, the floor is $H(p\otimes p) = 2 \times 1.75 = 3.5$ bits, and the overpayment is $\tfrac12$ bit, twice $\tfrac14$.

A bounded distance cannot do this: total variation never exceeds $1$, since $\sum_x|p_x - q_x| \le \sum_x p_x + \sum_x q_x = 2$, so it cannot grow like $n$.

**The chain rule.** For any two joint tables, write $p_{AB}(a,b) = p_A(a)\,p_{B|a}(b)$ and the same for $q$. The same expansion of the logarithm gives
$$
D(p_{AB}\|q_{AB}) = D(p_A\|q_A) + \sum_a p_A(a)\,D(p_{B|a}\|q_{B|a}):
$$
first the marginals, then the conditionals, averaged over the truth. With independent tables, additivity comes back. With $q$ uniform, each relative entropy is a $\log_2 n$ minus an entropy, and the chain rule becomes the chain rule of entropy of `02/01` §3, $H(AB) = H(A) + H(B|A)$.

*By hand on a correlated table.* Let $p_{AB} = \begin{pmatrix}0.4 & 0.1\\ 0.1 & 0.4\end{pmatrix}$ (two bits that agree $80\%$ of the time) and $q_{AB}$ uniform. Both marginals of $p$ are $(\tfrac12, \tfrac12)$, like those of $q$, so the marginal term is $0$. Each conditional of $p$ is $(0.8, 0.2)$ or $(0.2, 0.8)$ against $(\tfrac12, \tfrac12)$, which costs $1 - h(0.8)$, with $h(0.8) = 0.8\log_2\frac1{0.8} + 0.2\log_2\frac1{0.2}$ a little above $0.72$. So $D(p_{AB}\|q_{AB}) = 1 - h(0.8)$, a little below $0.28$ bits, all of it in the correlation. Let's check both computations, test the two laws on random tables, and draw $D$ and total variation against the number of symbols.

In [ ]:
# by hand 1: the dyadic source of 02/01, one symbol and two symbols at a time
p_dy = np.array([0.5, 0.25, 0.125, 0.125]); u4 = np.full(4, 0.25)
pp, uu = np.outer(p_dy, p_dy).ravel(), np.outer(u4, u4).ravel()
print(f"one symbol : D(p||u) = {kl_divergence(p_dy, u4):.4f} bits")
print(f"two symbols: D(p x p || u x u) = {kl_divergence(pp, uu):.4f} bits  = cost {-(pp * np.log2(uu)).sum():.2f} - floor H(p x p) {shannon_entropy(pp):.2f}")

# by hand 2: the chain rule on the 80%-agreement table against the uniform table
P = np.array([[0.4, 0.1], [0.1, 0.4]]); Q = np.full((2, 2), 0.25)
PA, QA = P.sum(axis=1), Q.sum(axis=1)
marginal_term = kl_divergence(PA, QA)
conditional_terms = [kl_divergence(P[a] / PA[a], Q[a] / QA[a]) for a in range(2)]
h08 = shannon_entropy([0.8, 0.2])
print(f"\nchain rule by hand: marginal term {marginal_term:.4f} + averaged conditionals {PA @ conditional_terms:.4f}  =  D(P||Q) {kl_divergence(P, Q):.4f};   1 - h(0.8) = 1 - {h08:.4f} = {1 - h08:.4f}")

# the two laws on random tables
pA, qA = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3))
pB, qB = rng.dirichlet(np.ones(2)), rng.dirichlet(np.ones(2))
print(f"\nrandom product tables: D(pA x pB || qA x qB) = {kl_divergence(np.outer(pA, pB), np.outer(qA, qB)):.6f}  =  D(pA||qA) + D(pB||qB) = {kl_divergence(pA, qA) + kl_divergence(pB, qB):.6f}")
PAB = rng.dirichlet(np.ones(6)).reshape(3, 2); QAB = rng.dirichlet(np.ones(6)).reshape(3, 2)
pA_, qA_ = PAB.sum(1), QAB.sum(1)
chain = kl_divergence(pA_, qA_) + sum(pA_[a] * kl_divergence(PAB[a] / pA_[a], QAB[a] / qA_[a]) for a in range(3))
print(f"random correlated 3x2 tables: D(P||Q) = {kl_divergence(PAB, QAB):.6f}  =  marginal term + averaged conditionals = {chain:.6f}")

# n independent copies of the 02/02 pair: relative entropy grows like n, total variation saturates
ns = np.arange(1, 11)
D_n, TV_n = [], []
pn, qn = np.ones(1), np.ones(1)
for n in ns:
    pn, qn = np.kron(pn, p), np.kron(qn, q)           # the product tables p^{(x) n}, q^{(x) n}: 3^n entries
    D_n.append(kl_divergence(pn, qn)); TV_n.append(tv(pn, qn))
D_n, TV_n = np.array(D_n), np.array(TV_n)
print(f"\nn copies:  D(p^n||q^n) / n = {np.round(D_n / ns, 4)}")
print(f"           TV(p^n, q^n)     = {np.round(TV_n, 4)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(ns, D_n, "o", color=viz.SOURCE_COLOR, ms=8, label="D(p^n || q^n) on the 3^n-entry tables")
ax.plot(ns, ns * kl_divergence(p, q), color=viz.SOURCE_COLOR, lw=1.5, ls="--", label=f"n D(p || q) = n x {kl_divergence(p, q):.4f}")
ax.set_xlabel("number n of independent symbols"); ax.set_ylabel("bits")
ax.set_title("Relative entropy adds up: a straight line", pad=10); ax.legend(loc="upper left")
ax = axes[1]
ax.plot(ns, TV_n, "o", color=viz.TARGET_COLOR, ms=8, label="TV(p^n, q^n)")
ax.plot(ns, ns * tv(p, q), color=viz.TARGET_COLOR, lw=1.5, ls="--", label=f"n TV(p, q) = n x {tv(p, q):.1f}  (what additivity would give)")
ax.axhline(1.0, color=COLORS["muted"], lw=1, ls=":")
ax.set_ylim(0, 1.2)
ax.set_xlabel("number n of independent symbols"); ax.set_ylabel("total variation")
ax.set_title("Total variation, a distance, saturates at 1", pad=10); ax.legend(loc="lower right")
fig.suptitle("Independent symbols: relative entropy grows at a fixed rate, a bounded distance cannot", y=1.02)
plt.show()

**Read the figure.** On the left, the dots are $D(p^{\otimes n}\|q^{\otimes n})$ computed directly on product tables of up to $3^{10} = 59{,}049$ entries, and they sit exactly on the dashed line $n \times 0.2651$ bits: each new independent symbol adds the same amount. On the right, total variation on the same tables rises and bends over, staying under the dotted ceiling at $1$ and far below the dashed line $n \times 0.3$ that additivity would demand; that line leaves the frame after four symbols.

**Read the output.** The dyadic pair costs $0.25$ bits for one symbol and $0.5$ for two, the $4$ bits per pair minus the floor of $3.5$. On the $80\%$-agreement table the marginal term is $0$ and the averaged conditionals give $0.2781 = 1 - 0.7219$ bits, equal to $D(P\|Q)$ computed directly. On random tables, additivity and the chain rule agree to six decimals. The ratio $D(p^{\otimes n}\|q^{\otimes n})/n$ is $0.2651$ for every $n$, while total variation climbs from $0.3$ to only $0.6835$ at $n = 10$.

## 2. One engine: the log-sum inequality, and joint convexity

Additivity came from the logarithm of a product. The next two laws come from one inequality about sums.

**The log-sum inequality.** For non-negative numbers $a_1, \dots, a_k$ and positive $b_1, \dots, b_k$, with totals $A = \sum_i a_i$ and $B = \sum_i b_i$,
$$
\sum_i a_i\log\frac{a_i}{b_i} \;\ge\; A\log\frac{A}{B},
$$
with equality if and only if all the ratios $a_i/b_i$ are equal. In words: lumping terms together can only lower the sum.

**Proof by hand.** Jensen's inequality of `02/01` §2 has a mirror image for a convex $f$: for weights $w_i \ge 0$ summing to one, $\sum_i w_i f(t_i) \ge f(\sum_i w_i t_i)$, since the chord now lies above the graph. When $f$ is strictly convex, equality holds only if all the $t_i$ coincide. Take $f(t) = t\log t$, strictly convex because $f''(t) = 1/(t\ln 2) > 0$ in bits, and the weights $b_i/B$:
$$
\sum_i a_i\log\frac{a_i}{b_i} = B\sum_i \frac{b_i}{B}\,f\Big(\frac{a_i}{b_i}\Big) \;\ge\; B\,f\Big(\sum_i \frac{b_i}{B}\frac{a_i}{b_i}\Big) = B\,f\Big(\frac{A}{B}\Big) = A\log\frac{A}{B}.
$$
Equality therefore needs all the points $a_i/b_i$ to coincide (Cover & Thomas, §2.7). With $A = B = 1$ the right-hand side is $0$: Gibbs' inequality once more.

*By hand on two terms.* Take the first two outcomes of `02/02`'s pair, $a = (0.7, 0.2)$ and $b = (0.4, 0.4)$. The left side is $0.7\log_2 1.75 + 0.2\log_2 0.5$, about $0.565 - 0.2 = 0.365$ bits; the right side is $0.9\log_2\frac{0.9}{0.8}$, about $0.153$ bits. The ratios $1.75$ and $0.5$ differ, so the inequality is strict. Take the last two outcomes instead, $a = (0.2, 0.1)$ and $b = (0.4, 0.2)$: both ratios are $0.5$, and both sides equal $-0.3$.

**Joint convexity.** For two pairs $(p_1, q_1)$, $(p_2, q_2)$ and a weight $\lambda \in [0, 1]$,
$$
D\big(\lambda p_1 + (1-\lambda)p_2\,\big\|\,\lambda q_1 + (1-\lambda)q_2\big) \;\le\; \lambda D(p_1\|q_1) + (1-\lambda)D(p_2\|q_2):
$$
mixing two sources and two models costs no more than the mixture of the costs. The proof is the log-sum inequality applied outcome by outcome, with the two terms $a_1 = \lambda p_1(x)$, $a_2 = (1-\lambda)p_2(x)$, $b_1 = \lambda q_1(x)$, $b_2 = (1-\lambda)q_2(x)$. The factors $\lambda$ and $1 - \lambda$ cancel inside the ratios $a_i/b_i$, so the left side of the log-sum bound is the $x$-term of the right side of joint convexity, and its right side is the $x$-term of the left. Sum over $x$.

Fix the model, $q_1 = q_2 = q$, and the same argument shows that $D(\cdot\|q)$ is convex in its first argument, in fact *strictly* convex, because $t\log t$ is. Total variation is jointly convex too, but not strictly: it is built from absolute values, which are straight on each side of zero, so minimising it under constraints can leave a whole segment of answers. A strictly convex function has at most one minimiser on a convex set, and that is what makes "minimise $D(\pi\|K)$ under linear constraints" a problem with a unique answer: the KL projections of `03/11` and `04/08`. Let's check the log-sum inequality, draw the convexity along a mixing path, and test joint convexity on two thousand random mixtures.

In [ ]:
def log_sum_sides(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    nz = a > 0
    lhs = float((a[nz] * np.log2(a[nz] / b[nz])).sum())
    rhs = float(a.sum() * np.log2(a.sum() / b.sum()))
    return lhs, rhs

for a, b in [((0.7, 0.2), (0.4, 0.4)), ((0.2, 0.1), (0.4, 0.2))]:
    lhs, rhs = log_sum_sides(a, b)
    print(f"log-sum on a = {a}, b = {b}: left {lhs:.4f}  >=  right {rhs:.4f}   gap {lhs - rhs:.4f}   (ratios a/b = {np.round(np.array(a) / np.array(b), 2)})")
worst_ls = np.inf
for _ in range(20000):
    k = rng.integers(2, 6)
    lhs, rhs = log_sum_sides(rng.uniform(0, 1, k), rng.uniform(0.01, 1, k))
    worst_ls = min(worst_ls, lhs - rhs)
print(f"log-sum on 20,000 random vectors: smallest left - right = {worst_ls:.3e}  (never negative)")

# joint convexity on 2000 random quadruples of four-outcome distributions
gaps_conv = []
for _ in range(2000):
    p1, p2, q1, q2 = rng.dirichlet(np.ones(4), size=4)
    lam = rng.uniform()
    lhs = kl_divergence(lam * p1 + (1 - lam) * p2, lam * q1 + (1 - lam) * q2)
    rhs = lam * kl_divergence(p1, q1) + (1 - lam) * kl_divergence(p2, q2)
    gaps_conv.append(rhs - lhs)
gaps_conv = np.array(gaps_conv)
print(f"joint convexity on 2000 random mixtures: smallest (mixture of D) - (D of mixtures) = {gaps_conv.min():.3e},  median {np.median(gaps_conv):.4f} bits")

# a mixing path between two pairs: (p, q) of 02/02 and a second pair (p2, q2)
p2_, q2_ = np.array([0.1, 0.3, 0.6]), np.array([0.3, 0.3, 0.4])
lams = np.linspace(0, 1, 201)
curve = np.array([kl_divergence(l * p + (1 - l) * p2_, l * q + (1 - l) * q2_) for l in lams])
chord = lams * kl_divergence(p, q) + (1 - lams) * kl_divergence(p2_, q2_)
first_arg = np.array([kl_divergence(l * p + (1 - l) * p2_, q) for l in lams])
first_chord = lams * kl_divergence(p, q) + (1 - lams) * kl_divergence(p2_, q)
print(f"mixing path: D(p||q) = {kl_divergence(p, q):.4f}, D(p2||q2) = {kl_divergence(p2_, q2_):.4f}; at lambda = 1/2 the curve is {curve[100]:.4f} against the chord {chord[100]:.4f}")
print(f"first argument only (model q fixed): D(p2||q) = {kl_divergence(p2_, q):.4f}; at lambda = 1/2 the curve is {first_arg[100]:.4f} against the chord {first_chord[100]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(lams, curve, color=viz.SOURCE_COLOR, lw=2.4, label="D(mixed truth || mixed model)")
ax.plot(lams, chord, color=viz.SOURCE_COLOR, lw=1.3, ls="--", label="mixture of the two costs (chord)")
ax.fill_between(lams, curve, chord, color=viz.SOURCE_COLOR, alpha=0.15)
ax.plot(lams, first_arg, color=viz.TARGET_COLOR, lw=2.4, label="D(mixed truth || q): model fixed")
ax.plot(lams, first_chord, color=viz.TARGET_COLOR, lw=1.3, ls="--", label="its chord")
ax.fill_between(lams, first_arg, first_chord, color=viz.TARGET_COLOR, alpha=0.15)
ax.set_xlabel("mixing weight lambda  (0: second pair, 1: the pair of 02/02)"); ax.set_ylabel("bits")
ax.set_title("Along a mixing path the cost stays below its chord", pad=10); ax.legend(loc="upper right")
ax = axes[1]
ax.hist(gaps_conv, bins=50, color=COLORS["highlight"], alpha=0.85)
ax.axvline(0.0, color=COLORS["text"], lw=1.2)
ax.set_xlabel("mixture of D  minus  D of the mixtures  (bits)"); ax.set_ylabel("number of random mixtures")
ax.set_title("2000 random mixtures: the gap is never negative", pad=10)
fig.suptitle("Joint convexity: mixing sources and models never costs more than mixing the costs", y=1.02)
plt.show()

**Read the figure.** On the left, the purple curve follows the cost as both the truth and the model slide from the second pair ($\lambda = 0$) to the pair of `02/02` ($\lambda = 1$). It sags below the straight chord between its endpoints all the way, and the shaded lens is the saving that mixing brings. The amber curve keeps the model fixed at $q$ and mixes only the truth; it bends below its own chord too, as a strictly convex function must, and has a single lowest point, the uniqueness `03/11` will need. On the right, the histogram of the gap between the two sides of joint convexity over two thousand random mixtures lies entirely to the right of the dark line at $0$.

**Read the output.** The log-sum inequality on $a = (0.7, 0.2)$, $b = (0.4, 0.4)$ gives $0.3651$ against $0.1529$ bits, a gap of $0.2122$ that section 3 meets again; with equal ratios both sides are $-0.3$. With this seed, over $20{,}000$ random vectors the smallest value of left minus right is $1.7\times10^{-7}$, positive, and over two thousand random mixtures the smallest gap of joint convexity is $5.1\times10^{-5}$ bits and the median $0.2782$: the inequality never fails and is usually far from tight. On the drawn path the midpoint costs $0.0335$ bits against $0.2288$ on the chord, and with the model fixed, $0.1131$ against $0.4458$.

## 3. Processing can only spend it: the data-processing inequality

A **stochastic map** $T$ sends a probability vector to a probability vector: $(Tp)_y = \sum_x T_{yx}\,p_x$, with $T_{yx} \ge 0$ and every column summing to one. It is the classical noisy channel of `01/20`: input $x$ becomes output $y$ with probability $T_{yx}$. Merging outcomes, adding noise, relabelling and forgetting are all such maps.

**Claim.** $D(Tp\|Tq) \le D(p\|q)$ for every stochastic map $T$.

**Proof by hand.** Apply the log-sum inequality to each output $y$ with the terms $a_x = T_{yx}p_x$ and $b_x = T_{yx}q_x$, then sum over $y$:
$$
D(Tp\|Tq) = \sum_y \Big(\sum_x T_{yx}p_x\Big)\log_2\frac{\sum_x T_{yx}p_x}{\sum_x T_{yx}q_x}
\;\le\; \sum_y\sum_x T_{yx}\,p_x\log_2\frac{p_x}{q_x} = \sum_x p_x\log_2\frac{p_x}{q_x} = D(p\|q),
$$
where the factors $T_{yx}$ cancel inside each ratio $a_x/b_x$, and the last step uses $\sum_y T_{yx} = 1$. Processing, deterministic or random, can never make two sources look less alike.

*By hand: two merges.* Merge outcomes $1$ and $2$ of `02/02`'s pair: $p \mapsto (0.9, 0.1)$ and $q \mapsto (0.8, 0.2)$. The merged cell contributes $0.9\log_2\frac{0.9}{0.8}$ where the two original cells contributed $0.7\log_2\frac{0.7}{0.4} + 0.2\log_2\frac{0.2}{0.4}$, so the loss is exactly the log-sum gap of section 2, $0.2122$ bits. Merge outcomes $2$ and $3$ instead: $p \mapsto (0.7, 0.3)$ and $q \mapsto (0.4, 0.6)$. Both ratios $p_x/q_x$ were $0.5$, the log-sum gap is $0$, and the merge loses nothing. That is the equality case for merges: a merge keeps the whole cost exactly when the merged outcomes share the same ratio $p_x/q_x$. A permutation of the outcomes is a stochastic map whose inverse is one too, so the inequality holds both ways and relabelling keeps $D$ exactly, as `02/02` §3 observed.

*Noise that grows.* Let $T_\eta = (1 - \eta)\,I + \eta\,U$, where every column of $U$ is $(\tfrac13, \tfrac13, \tfrac13)$: with probability $\eta$ the symbol is replaced by a random one. Because $UI = IU = UU = U$, two such maps compose into a third, $T_a T_b = T_{1 - (1-a)(1-b)}$. So a noisier map is a less noisy one followed by more noise (for $\eta' > \eta$, $T_{\eta'} = T_b T_\eta$ with $b = \frac{\eta' - \eta}{1 - \eta}$), and the inequality forces $D(T_\eta p\|T_\eta q)$ to decrease as $\eta$ grows. Let's check the inequality on random maps, the two merges, and the noise family.

In [ ]:
# random stochastic maps: columns are probability vectors
ratios = []
for _ in range(2000):
    T = rng.dirichlet(np.ones(3), size=3).T            # 3x3, each column a probability vector
    ratios.append(kl_divergence(T @ p, T @ q) / kl_divergence(p, q))
ratios = np.array(ratios)
print(f"2000 random stochastic T: largest D(Tp||Tq) / D(p||q) = {ratios.max():.4f}, median {np.median(ratios):.4f}  (never above 1)")

T_merge12 = np.array([[1, 1, 0], [0, 0, 1]], dtype=float)   # outcomes 1 and 2 become one
T_merge23 = np.array([[1, 0, 0], [0, 1, 1]], dtype=float)   # outcomes 2 and 3 become one
T_perm = np.eye(3)[[2, 0, 1]]                                # a relabelling
D0 = kl_divergence(p, q)
for name, T in [("merge 1+2", T_merge12), ("merge 2+3", T_merge23), ("relabel", T_perm)]:
    print(f"{name:10s}: p -> {T @ p}, q -> {T @ q};  D(p||q) = {D0:.4f}  ->  D(Tp||Tq) = {kl_divergence(T @ p, T @ q):.4f}   loss {D0 - kl_divergence(T @ p, T @ q):.4f}")
print(f"ratios p/q of the merged outcomes: 1+2 -> {p[0] / q[0]:.2f}, {p[1] / q[1]:.2f};  2+3 -> {p[1] / q[1]:.2f}, {p[2] / q[2]:.2f}")

# the noise family T_eta = (1 - eta) I + eta U
U = np.full((3, 3), 1 / 3)
def T_eta(eta):
    return (1 - eta) * np.eye(3) + eta * U
etas = np.linspace(0, 1, 201)
D_eta = np.array([kl_divergence(T_eta(e) @ p, T_eta(e) @ q) for e in etas])
print(f"composition check: T_0.3 T_0.5 = T_{1 - 0.7 * 0.5:.2f}:  max entry difference {np.abs(T_eta(0.3) @ T_eta(0.5) - T_eta(0.65)).max():.1e}")
print(f"noise family: D = {D_eta[0]:.4f} at eta = 0, {D_eta[50]:.4f} at 0.25, {D_eta[100]:.4f} at 0.5, {D_eta[150]:.4f} at 0.75, {D_eta[-1]:.4f} at 1;  monotone decreasing: {bool(np.all(np.diff(D_eta) <= 1e-15))}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(etas, D_eta, color=viz.SOURCE_COLOR, lw=2.4, label="D(T_eta p || T_eta q)")
ax.axhline(D0, color=COLORS["muted"], lw=1, ls="--", label=f"D(p || q) = {D0:.4f}, before any processing")
ax.set_xlabel("noise level eta  (chance of a random replacement)"); ax.set_ylabel("bits")
ax.set_title("More noise, less distinguishability", pad=10); ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.9))
ax = axes[1]
ax.hist(ratios, bins=40, range=(0, 1.05), color=COLORS["highlight"], alpha=0.85, label="2000 random stochastic maps")
for name, T, col in [("merge 1+2", T_merge12, viz.TARGET_COLOR), ("merge 2+3 and relabel", T_merge23, viz.SOURCE_COLOR)]:
    ax.axvline(kl_divergence(T @ p, T @ q) / D0, color=col, lw=2, ls="--", label=name)
ax.axvline(1.0, color=COLORS["text"], lw=1.2)
ax.set_xlabel("D(Tp || Tq) / D(p || q)"); ax.set_ylabel("number of maps")
ax.set_title("No map pushes the ratio past 1", pad=10); ax.legend(loc="upper left", bbox_to_anchor=(0.25, 1.0))
fig.suptitle("The data-processing inequality: processing can only spend distinguishability", y=1.02)
plt.show()

**Read the figure.** On the left, the divergence after the noise $T_\eta$ starts at the full $0.2651$ bits and falls as the noise grows, reaching $0$ when every symbol is replaced at random, and it never rises again. On the right, the ratio $D(Tp\|Tq)/D(p\|q)$ over two thousand random stochastic maps spreads mostly below $0.4$, because a random map mixes the outcomes heavily, and never reaches the dark line at $1$. The amber dashed line is the merge of outcomes $1$ and $2$, at a fifth of the original cost.

**Read the output.** No random map reached the bound: with this seed the largest ratio is $0.8749$ and the median $0.1538$. Merging outcomes $1$ and $2$ drops $0.2651$ bits to $0.0529$, a loss of $0.2122$, the log-sum gap of section 2: the model's two mistakes ($0.7$ against $0.4$, $0.2$ against $0.4$) partly cancel once the two outcomes are indistinguishable. Merging outcomes $2$ and $3$ loses nothing, because there $p/q$ was $0.50$ on both, and neither does the relabelling. The composition rule holds to machine precision, and along the noise family the divergence decreases monotonically: $0.2651$, $0.1496$, $0.0682$, $0.0178$ and $0$ at $\eta = 0, 0.25, 0.5, 0.75, 1$. Distinguishability is a resource that processing can only spend.

## 4. Where the parent quantity goes

Three objects of the course are relative entropies in disguise, and each inherits a law proved today.

- **Mutual information** (`02/04`): how far is a joint table from the product of its marginals? $I(A{:}B) = D(p_{AB}\,\|\,p_A \otimes p_B)$. Gibbs (`02/02`) gives $I \ge 0$, with equality if and only if the variables are independent. By the chain rule of section 1 it is also $\sum_a p_A(a)\,D(p_{B|a}\|p_B)$: how much, on average, learning $a$ moves the distribution of $b$. By section 3, processing one variable can only lower it.
- **Entropic optimal transport** (`03/09`–`03/11`): Sinkhorn's plan is the table $\pi$ with prescribed marginals that minimises $D(\pi\,\|\,K)$, in nats, to a Gibbs kernel $K = e^{-C/\varepsilon}$: a **KL projection** in the sense of Csiszár (1975), with the transport reading in `03/11`. The kernel is a positive table that does not sum to one. The same sum still makes sense but can be negative, since Gibbs' inequality needs both arguments to sum to one. The library's `kl_to_kernel`, used in `03/11`, adds $\sum K - \sum \pi$ to restore non-negativity, and on tables of fixed mass that addition is a constant, so the minimiser does not move. The feasible plans form a compact convex set, so a minimiser exists, and the strict convexity of section 2 makes it the only one.
- **Umegaki relative entropy** (`02/10`): replace $p, q$ by density matrices, $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$. When $\rho$ and $\sigma$ are diagonal in the same basis it *is* $D$ of the diagonals. It inherits non-negativity (Klein's inequality), the infinite value off the support, and monotonicity under every quantum channel. That last property is section 3, lifted, and a deep theorem (Lindblad 1975; Uhlmann 1977 in full generality): today's log-sum proof does not carry over to matrices that do not commute. What the lift adds is the coherence: $|{+}\rangle$ and $I/2$ share the diagonal $(\tfrac12, \tfrac12)$, so the classical $D$ of their $Z$-basis statistics is $0$, while $S(|{+}\rangle\langle{+}|\,\|\,I/2)$ is a full bit, the seed of `01/14` finally seen by a relative entropy.

Let's compute the two numbers this list quotes and the commuting case of the third.

In [ ]:
# mutual information as a relative entropy, on two tables of 01/17
coins = np.array([[0.5, 0.0], [0.0, 0.5]])            # the correlated coins: 00 and 11, each with probability 1/2
indep = np.outer([0.5, 0.5], [0.5, 0.5])             # the independent table
for name, J in [("correlated coins", coins), ("independent table", indep)]:
    prod = np.outer(J.sum(axis=1), J.sum(axis=0))      # the product of the marginals
    print(f"{name:18s}: D(p_AB || p_A x p_B) on the flattened table = {kl_divergence(J.ravel(), prod.ravel()):.4f} bits;  helper mutual_information = {mutual_information(J):.4f}")

# Umegaki: the commuting case is the classical D; the coherent pair is not
rho_diag, sigma_diag = np.diag(p), np.diag(q)
print(f"\ndiagonal states diag(p), diag(q): S = {quantum_relative_entropy(rho_diag, sigma_diag):.4f} bits  =  D(p||q) = {kl_divergence(p, q):.4f}")
plus = density_matrix(KET_PLUS); mixed = maximally_mixed(2)
print(f"|+><+| against I/2: classical D of the Z-basis diagonals = {kl_divergence(np.real(np.diag(plus)), np.real(np.diag(mixed))):.4f};  Umegaki S = {quantum_relative_entropy(plus, mixed):.4f} bits")

**Read the output.** On the correlated coins the relative entropy of the flattened table to the product of its marginals is $1$ bit, and on the independent table $0$, in agreement with the helper `mutual_information` that `02/04` will use. On diagonal states the Umegaki relative entropy returns the classical $0.2651$ bits of `02/02`'s pair: in the commuting case nothing new happens. On $|{+}\rangle$ against $I/2$ the classical value is $0$ and the Umegaki value $1$ bit: the coherence is invisible to the diagonal and plain to the quantum quantity. The classical column of the dictionary's relative-entropy row was earned in `02/02`; today you gave it the three laws its quantum column must keep.

## Your turn

1. **The chain rule, both ways.** Write any $3\times3$ joint table $P$ and any $3\times3$ model $Q$ (normalise both). Compute $D(P\|Q)$, then the chain-rule decomposition "$A$ first, then $B$ given $A$" and "$B$ first, then $A$ given $B$". Both give the same total; are their two terms the same? Find a pair $(P, Q)$ whose marginal term is $0$ in one order and not in the other.
2. **Gibbs from the log-sum inequality, and merges that lose nothing.** (a) With $A = B = 1$ the log-sum inequality is Gibbs' inequality (section 2). Use its equality case to show that equality forces $p = q$, including when $q$ puts mass where $p$ has none. (b) For $p = (0.4, 0.2, 0.1, 0.3)$ and $q = (0.2, 0.1, 0.175, 0.525)$, find every way of merging the four outcomes into two groups that keeps $D(p\|q)$ unchanged, using only the ratios $p_x/q_x$; then confirm numerically.
3. **A noisy coin, and what the inequality does not say.** Send $p = (0.9, 0.1)$ and $q = (0.6, 0.4)$ through the binary symmetric channel that flips the bit with probability $\epsilon$, and plot $D(Tp\|Tq)$ against $\epsilon \in [0, 1]$. Where is it $0$? Why is the curve symmetric about $\epsilon = \tfrac12$, and why does the data-processing inequality not forbid it from *rising* between $\epsilon = \tfrac12$ and $\epsilon = 1$? (Hint: is the channel with $\epsilon = 1$ the channel with $\epsilon = \tfrac12$ followed by another channel?)
4. **Harder: convexity from processing.** (a) Attach a label to each pair: on the outcomes $(i, x)$ with $i \in \{1, 2\}$, let $\tilde p(i, x) = \lambda_i\,p_i(x)$ and $\tilde q(i, x) = \lambda_i\,q_i(x)$, with $\lambda_1 = \lambda$ and $\lambda_2 = 1 - \lambda$. Use the chain rule to show $D(\tilde p\|\tilde q) = \lambda D(p_1\|q_1) + (1-\lambda)D(p_2\|q_2)$. (b) The map that forgets the label, $(i, x) \mapsto x$, is stochastic. Apply the data-processing inequality to it and obtain joint convexity in one line. Check the whole argument numerically on one random quadruple.

## What you built

- You can prove that relative entropy adds up over independent symbols and splits by the chain rule, compute both by hand (two dyadic symbols overpay $\tfrac12$ bit; the $80\%$-agreement table costs $1 - h(0.8) = 0.2781$ bits, all in the correlation), and explain why no bounded distance is additive: $0.2651$ bits per symbol accumulate while total variation stays below $1$.
- You can prove the log-sum inequality from Jensen's inequality, with its equality case, and derive from it Gibbs' inequality again, joint convexity, and the strict convexity that makes a KL projection unique.
- You can prove the data-processing inequality, predict by hand which merges lose nothing (equal ratios $p_x/q_x$) and how much the others lose (the log-sum gap, $0.2122$ bits for the first merge of `02/02`'s pair), and show that growing noise makes the divergence decrease.
- You can name the three relative entropies ahead and the law each one inherits: mutual information, the Sinkhorn projection and Umegaki's lift.

The log-sum inequality carried two of today's three laws, and the logarithm of a product carried the third.

## What's next

Point relative entropy at a joint table and its own product of marginals, and it answers a new question: how much do two variables share? That is the mutual information of `02/04_mutual_information`, zero exactly when the table factorises, and the classical cousin of the quantum mutual information that reads the Bell state in `02/11`.

## References

- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.5 "Chain Rules for Entropy, Relative Entropy, and Mutual Information", §2.7 "Log Sum Inequality and Its Applications", §2.8 "Data-Processing Inequality". doi:10.1002/047174882X
- S. Kullback & R. A. Leibler, *On information and sufficiency*, Ann. Math. Statist. **22**, 79–86 (1951) — the definition. doi:10.1214/aoms/1177729694
- I. Csiszár, *Information-type measures of difference of probability distributions and indirect observations*, Studia Sci. Math. Hungar. **2**, 299–318 (1967).
- I. Csiszár, *I-divergence geometry of probability distributions and minimization problems*, Ann. Probab. **3**, 146–158 (1975) — I-projections onto linear families; `03/11` credits the Sinkhorn reading to Cuturi (2013) and Benamou et al. (2015). doi:10.1214/aop/1176996454
- S. Amari & H. Nagaoka, *Methods of Information Geometry*, Translations of Mathematical Monographs **191**, AMS/Oxford (2000) — the information geometry behind the bridge of `03/11`. doi:10.1090/mmono/191
- H. Umegaki, *Conditional expectation in an operator algebra, IV (entropy and information)*, Kodai Math. Sem. Rep. **14**, 59–85 (1962) — the quantum relative entropy of `02/10`. doi:10.2996/kmj/1138844604
- G. Lindblad, *Completely positive maps and entropy inequalities*, Commun. Math. Phys. **40**, 147–151 (1975) — monotonicity of the Umegaki relative entropy under channels, the quantum section 3. doi:10.1007/BF01609396
- A. Uhlmann, *Relative entropy and the Wigner–Yanase–Dyson–Lieb concavity in an interpolation theory*, Commun. Math. Phys. **54**, 21–32 (1977) — monotonicity in full generality. doi:10.1007/BF01609834

**Previous:** `notebooks/02_InformationAndGeometry/02_kl_divergence.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/04_mutual_information.ipynb`